In [15]:
!pip3 install pandas numpy matplotlib scikit-learn

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 7.8 MB 4.3 MB/s eta 0:00:01
     |████████████████████████████████| 11.1 MB 19.2 MB/s eta 0:00:01
     |████████████████████████████████| 249 kB 38.7 MB/s eta 0:00:01
     |████████████████████████████████| 2.9 MB 57.7 MB/s eta 0:00:01
     |████████████████████████████████| 4.7 MB 34.8 MB/s eta 0:00:01
     |████████████████████████████████| 64 kB 19.6 MB/s eta 0:00:01
     |████████████████████████████████| 122 kB 41.3 MB/s eta 0:00:01
     |████████████████████████████████| 309 kB 102.9 MB/s eta 0:00:01
     |████████████████████████████████| 30.3 MB 18.6 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [16]:
import pandas as pd

In [17]:
fake = pd.read_csv("data/Fake.csv")
true = pd.read_csv("data/True.csv")

fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [19]:
fake["label"] = 0
true["label"] = 1

fake.head()
#true.head()

,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0


In [20]:
df = pd.concat([fake, true]).sample(frac=1).reset_index(drop=True)
df["text"] = df["title"] + " " + df["text"]
df = df[["text", "label"]]

df.head()

,text,label
0,Sudan to extend ceasefire through end-December...,1
1,Germany balks at Tillerson call for more Europ...,1
2,EU's Barnier paints gloom and doom for post-Br...,1
3,FATHERS OF SONS MURDERED By Illegal Aliens Hav...,0
4,WATCH DEMOCRATS Repeat Exact Same Phrase On Fa...,0


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(stop_words="english", max_features=10000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

print(classification_report(y_test, model.predict(X_test_vec)))

# Save model
pickle.dump((vectorizer, model), open("model/model.pkl", "wb"))

/Users/honeypatel/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/honeypatel/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: overflow encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/honeypatel/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: invalid value encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)


              precision    recall  f1-score   support

           0       0.99      0.98      0.99      4742
           1       0.98      0.99      0.99      4238

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980

